# Laboratorio 7 — Visualización de datos: EDA y reportería

IA7202: Laboratorio de Programación Científica para Ciencia de Datos - Otoño 2026

---

### Cuerpo Docente

- Profesor: Pablo Badilla y Ignacio Nuñez
- Auxiliar: Sofía Chávez
- Ayudantes: Javiera Arévalo, Tamara Carrasco, Ignacio Reyes

---

### Equipo

Escriban los nombres de ambos integrantes. No se revisarán entregas sin esta
identificación.

- Nombre de alumno 1:
- Nombre de alumno 2:

---

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Reglas del laboratorio</strong>
  El trabajo es <strong>en parejas</strong> y la entrega es la pull request con el CI en verde y mergeada.
</div>

## Flujo de trabajo

La Parte 1 sigue el mismo ciclo de principio a fin: leer la sección,
ejecutar sus celdas, construir el gráfico que se pide y responder la
pregunta con lo que observan. Los entregables de código dicen qué gráfico
quieren y con qué condiciones, pero **no traen el llamado escrito**: la
función y sus argumentos salen de la documentación de Plotly, que va
enlazada en el recuadro de definición anterior a cada uno.

La Parte 2 se arma aparte, fuera del notebook: un panel de Streamlit con su
tema en `.streamlit/config.toml`. El laboratorio les entrega los dos
archivos empezados —`mi_panel.py` con la infraestructura escrita y las
cuatro secciones vacías, y el `config.toml` con las claves comentadas—. No
hay panel de ejemplo resuelto: los requisitos están en su entregable y las
decisiones son suyas.

## Configuración del ambiente

Desde la carpeta del laboratorio, instalen con `uv` las librerías declaradas
en `pyproject.toml`:

```bash
uv sync
```

Ese archivo ya incluye `polars`, `pyarrow`, `plotly`, `statsmodels` y
`streamlit`; no necesitan agregarlos de nuevo. Si arman un proyecto propio
desde cero, declaren sus dependencias con:

```bash
uv add polars pyarrow plotly statsmodels "streamlit[charts]"
```

El paquete se llama `statsmodels`, con **s** al final. La opción
`streamlit[charts]` agrega las dependencias de gráficos de Streamlit;
`plotly` aparece también en el comando porque se usa directamente en el
notebook y en el panel.

## Contexto

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 El encargo — la prueba de Estudio Cifra</strong>
  Marisela Ipinza dirige Estudio Cifra, una consultora de tres personas en Providencia que arma reportes y paneles para pymes que no tienen a nadie propio mirándoles los números. Antes de dejar que alguien nuevo toque los datos de un cliente real, les toma la misma prueba a todos: un dataset chico y conocido, donde cualquier gráfico engañoso o cualquier conclusión apurada salta a la vista de inmediato.
  <br><br>
  Esta semana la prueba les toca a ustedes, con los datos de los pingüinos del archipiélago de Palmer. Primero revisarán los gráficos y explorarán la tabla. Después construirán un panel con estos mismos datos, para que otra persona pueda examinarlos desde el navegador.
  <br><br>
  Marisela les pide conservar el dataset original. Los datos se miran como llegan, con sus valores faltantes y sus categorías inesperadas. Si una figura necesita tratar un faltante para poder dibujarse, háganlo en una tabla aparte y documenten la decisión. No modifiquen <code>df</code>.
</div>

## Objetivos

- Distinguir un gráfico exploratorio de uno explicativo, y reconocer cuándo
  un gráfico exagera lo que muestra.
- Construir diez visualizaciones con Plotly Express leyendo su
  documentación, en vez de copiar una receta.
- Explorar un dataset tal como llega: forma, tipos, valores faltantes,
  composición, distribuciones y correlaciones.
- Reconocer qué hace cada herramienta con los datos que le faltan, que no
  es lo mismo en todas.
- Construir un panel local de Streamlit con tema propio y una tabla que el
  lector pueda ordenar y filtrar sin saber Python.

## Evaluación

<!-- EVALUACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8b5cf6; background:rgba(139,92,246,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📊 Cómo se calcula la nota</strong>
  El laboratorio tiene un máximo de <strong>6,0 puntos</strong>. La nota se calcula como <code>nota = 1,0 + puntaje obtenido</code>: 0,0 puntos corresponden a nota 1,0; 3,0 puntos a nota 4,0; y 6,0 puntos a nota 7,0.
</div>

| Parte | Contenido | Puntaje |
|---|---|---:|
| 1 | Visualización y análisis exploratorio | 3,0 |
| 2 | Dashboard | 3,0 |
| **Total** | | **6,0** |

## Preparación

Las dos partes trabajan sobre el mismo dataset, con `polars` y con `plotly`, la librería de graficado del curso. La Parte 2 vive en un archivo `.py` aparte, porque un panel de Streamlit no corre dentro de una celda de notebook: se lanza con `streamlit run`.

### Con qué se grafica en Python

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — librerías imperativas y declarativas</strong>
  Una librería de gráficos <strong>imperativa</strong> recibe órdenes de dibujo: creen una figura, agreguen unos ejes, dibujen estos puntos, pónganle una leyenda. Es <code>matplotlib</code>, y con ella el gráfico se construye paso a paso.
  <br><br>
  Una librería <strong>declarativa</strong> recibe una descripción: esta tabla, esta columna en X, esta en Y, esta otra como color. La librería decide cómo dibujarlo. Es <code>plotly.express</code>, y también <code>seaborn</code> o <code>altair</code>.
  <br><br>
  La diferencia práctica está en cómo se construye el gráfico. Con una librería imperativa indicamos los pasos de dibujo; con <code>plotly.express</code> podemos pedir, por ejemplo, <code>color="species"</code> para separar las especies. La librería propone colores, ejes y orden de categorías que después podemos ajustar.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — Plotly Express (<code>px</code>)</strong>
  <code>plotly.express</code> es la capa declarativa de Plotly. Una función por tipo de gráfico —<code>px.scatter</code>, <code>px.histogram</code>, <code>px.box</code>—, todas con la misma forma: la tabla primero, y después qué columna cumple cada rol.
  <br><br>
  Lo que lo hace cómodo en ciencia de datos son tres cosas concretas:
  <ol>
    <li><strong>Consume la tabla directo.</strong> Los argumentos son nombres de columna, no arreglos que haya que extraer y alinear a mano.</li>
    <li><strong>Los roles visuales son argumentos.</strong> <code>color</code>, <code>size</code>, <code>symbol</code> y <code>facet_col</code> mapean una columna a una propiedad visual. Agregar una variable al gráfico es agregar un argumento, no reescribirlo.</li>
    <li><strong>Sale interactivo.</strong> Zoom, <em>hover</em> con los valores y leyenda que filtra al hacer clic, sin configurar nada. Explorando, eso importa más que en un gráfico impreso.</li>
  </ol>
  Cuando no se le indica otra cosa, <code>px</code> elige la paleta, el rango de los ejes y el orden de las categorías.
  <br><br>
  Punto de partida de la documentación: <a href="https://plotly.com/python/plotly-express/">plotly.com/python/plotly-express</a>.
</div>

<!-- TIP -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2f81f7; background:rgba(47,129,247,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💡 Cómo leer la documentación de Plotly</strong>
  Cada página de <code>plotly.com/python</code> es una galería: ejemplos ejecutables de menos a más, y el código al lado de cada figura. Sirve para encontrar <em>qué</em> función usar.
  <br><br>
  Para saber <em>qué argumentos acepta</em>, la referencia de la API es otra: <a href="https://plotly.com/python-api-reference/generated/plotly.express.scatter.html">plotly.com/python-api-reference</a>, con una página por función y la lista completa de parámetros. Cuando un ejercicio pida algo que el ejemplo de la galería no muestra, está ahí.
</div>

In [ ]:
import sys
from pathlib import Path

RUTA_LAB = Path.cwd().parent
if str(RUTA_LAB) not in sys.path:
    sys.path.insert(0, str(RUTA_LAB))

RAW_DIR = RUTA_LAB / "data" / "raw"

---
## Parte 1 — Visualización y análisis exploratorio (3,0 puntos)

Al construir un gráfico elegimos qué variables representar en los ejes y si
usaremos color, forma o tamaño para mostrar otras.

En esta parte combinaremos gráficos con recuentos y resúmenes para explorar
los datos. Primero veremos cómo las decisiones visuales afectan la lectura;
después recorreremos el dataset para comprobar qué muestran los gráficos y
qué dejan fuera.

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 La carpeta del postulante anterior</strong>
  Marisela les deja sobre la mesa el CSV de los pingüinos y una carpeta con gráficos que le mandó el postulante del proceso pasado. «No alcancé a contratarlo», dice, «pero guardé lo que me entregó, porque me sirve para esto». Los gráficos de la carpeta no están rotos: abren, tienen datos reales y se ven prolijos. El problema es otro, y es el que Marisela quiere que le nombren.
  <br><br>
  Después viene la parte que a ella le importa más. Un cliente le devolvió un informe la semana pasada reclamando una cifra que no le calzaba con lo que sabía de su propio negocio, y tenía razón. «El gráfico estaba bien hecho», dice. «El problema es que nadie había mirado la tabla antes de graficarla.»
  <br><br>
  Así que el encargo es doble: arreglar los gráficos del postulante, y después recorrer el dataset entero hasta poder decir qué tiene, qué le falta y qué llegó escrito de una forma que nadie esperaba.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Los cuatro principios</strong>
  Al revisar los gráficos de este laboratorio, usaremos cuatro criterios:
  <ol>
    <li><strong>Definir qué se quiere mostrar:</strong> un gráfico puede servir para buscar un patrón o para comunicar un hallazgo.</li>
    <li><strong>Usar solo los elementos necesarios:</strong> agregar color, tamaño o forma cuando ayuden a leer los datos.</li>
    <li><strong>Elegir el gráfico según la pregunta:</strong> no todos permiten comparar lo mismo.</li>
    <li><strong>Separar cuando haga falta:</strong> varios paneles pueden ser más claros que muchas variables en uno solo.</li>
  </ol>
</div>

### Pingüinos de Palmer

Trabajan con datos de 344 pingüinos de tres especies (*Adelie*, *Chinstrap*, *Gentoo*), medidos en tres islas del archipiélago de Palmer, Antártica. Publicados por la Dra. Kristen Gorman y Palmer Station LTER.

El archivo se lee con `null_values=["NA"]` porque viene de R, donde `NA` es el marcador de dato faltante. Sin ese argumento, `polars` lee las columnas numéricas como texto. Y con él, los faltantes quedan adentro como nulos: **no los vamos a sacar**.

| Columna | Qué mide |
|---|---|
| `species` | especie del pingüino |
| `island` | isla de origen (`Torgersen`, `Biscoe`, `Dream`) |
| `culmen_length_mm` | largo del pico, en mm |
| `culmen_depth_mm` | alto del pico, en mm |
| `flipper_length_mm` | largo de la aleta, en mm |
| `body_mass_g` | masa corporal, en g |
| `sex` | sexo del pingüino |

In [ ]:
import polars as pl
import plotly.express as px

df = pl.read_csv(RAW_DIR / "penguins.csv", null_values=["NA"])
print(f"{df.height} pingüinos, {df['species'].n_unique()} especies")
df.head()

### Contar una historia

Un gráfico **exploratorio** ayuda a buscar patrones. Uno **explicativo**
comunica un hallazgo a una audiencia: selecciona qué mostrar y hace explícita
la interpretación. Aquí compararemos dos versiones de los mismos datos para
ver cómo cambian su lectura el color y el título.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — el gráfico de dispersión</strong>
  Un <strong>scatter</strong> pone una observación por punto y dos variables numéricas en los ejes. Es la herramienta por omisión para preguntar si dos medidas se mueven juntas.
  <br><br>
  En <code>px</code> se pide con <code>px.scatter(tabla, x=..., y=...)</code>, y desde ahí se le pueden agregar roles visuales: <code>color</code>, <code>size</code>, <code>symbol</code>. Documentación: <a href="https://plotly.com/python/line-and-scatter/">plotly.com/python/line-and-scatter</a>.
</div>

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable 1 — El mismo gráfico, dos veces (0,2 puntos)</strong>
  Construyan el mismo scatter dos veces, con <code>flipper_length_mm</code> en X y <code>body_mass_g</code> en Y sobre <code>df</code>:
  <ol>
    <li><code>fig_exploratorio</code>: solo los dos ejes. Nada más.</li>
    <li><code>fig_explicativo</code>: lo mismo, más <code>color</code> por especie y un <code>title</code> que <strong>diga en palabras el patrón</strong> — no «masa vs. aleta», sino la frase que ustedes le mandarían a Marisela.</li>
  </ol>
  Los dos en la misma celda, y muestren ambos. La gracia está en que difieran <strong>solo</strong> en esas dos cosas: si además les cambian el tamaño o los ejes, la comparación deja de servir.
</div>

In [ ]:
# Su código aquí


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 1 — Exploratorio vs. explicativo (0,2 puntos)</strong>
  <ol>
    <li>Miren los dos gráficos de arriba: ¿qué cambió entre uno y otro, y cuál es exploratorio y cuál es explicativo? Justifiquen con lo que ven, no con la definición.</li>
    <li>El segundo gráfico separa los puntos por <code>species</code>. Mirando la tabla de columnas de más arriba, ¿qué otra variable permitiría comparar grupos dentro de la relación entre aleta y masa corporal? ¿Qué tendrían que mirar para saber si la separación aporta información?</li>
  </ol>
</div>

`Escriban sus respuestas aquí:`

### Cuidado con el color

Para comparar magnitudes, suele ser más fácil leer posiciones sobre un eje
común que estimar tamaños o diferencias de color. Para distinguir categorías
también podemos usar color o forma. En el siguiente gráfico, quince colores
hacen difícil ordenar una variable que ya tiene valores numéricos.

In [ ]:
minimo, maximo = df["body_mass_g"].min(), df["body_mass_g"].max()
cortes = [minimo + i * (maximo - minimo) / 15 for i in range(1, 15)]

df_binned = df.with_columns(
    pl.col("body_mass_g").cut(cortes).alias("masa_binned")
)

fig_colores = px.scatter(
    df_binned,
    x="flipper_length_mm",
    y="culmen_length_mm",
    color="masa_binned",
)
fig_colores.show()

Una **escala continua** muestra los valores mediante un degradado ordenado.
Una escala secuencial va de menor a mayor; una divergente distingue valores
a ambos lados de una referencia, y una cíclica sirve para valores periódicos.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 2 — Quince colores para una variable (0,2 puntos)</strong>
  <ol>
    <li><code>body_mass_g</code> es una variable numérica continua. ¿Qué problema tiene representarla partiéndola en 15 categorías de color, como hace el gráfico de arriba?</li>
    <li>¿Qué deberían haber usado en su lugar, dado que es una variable continua y no categórica?</li>
    <li>Averigüen cuántos pingüinos quedaron fuera del gráfico y compárenlo con las 344 filas del archivo. ¿Cuántos faltan? Revisen qué valores les faltan a esas filas antes de explicar por qué no aparecen. (Hint: la respuesta no se obtiene contando puntos.)</li>
  </ol>
</div>

`Escriban sus respuestas aquí:`

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — escalas de color continuas</strong>
  Cuando la columna que pasan a <code>color</code> es numérica, <code>px</code> usa una <strong>escala continua</strong>: un degradado ordenado, con una barra que relaciona cada color con su valor. Así se puede leer el orden de las masas sin buscar quince categorías en la leyenda.
  <br><br>
  Para la masa corporal conviene una escala secuencial, porque sus valores se ordenan de menor a mayor y no tienen un punto central que separe dos sentidos. La paleta se elige con <code>color_continuous_scale</code>. Catálogo: <a href="https://plotly.com/python/colorscales/">plotly.com/python/colorscales</a>.
</div>

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable 2 — Arreglen el gráfico (0,2 puntos)</strong>
  Rehagan el gráfico de arriba como debió estar hecho: mismos ejes —<code>flipper_length_mm</code> y <code>culmen_length_mm</code>—, pero con <code>body_mass_g</code> en una escala continua en vez de los quince bins.
  <ul>
    <li>Trabajen sobre <code>df</code>, sin <code>cut</code> y sin crear columnas nuevas.</li>
    <li>Elijan una escala secuencial del catálogo, pásenla por <code>color_continuous_scale</code> y dejen en un comentario por qué corresponde a la masa corporal.</li>
    <li>Etiqueten con su unidad las dos variables de los ejes y la de la escala de color, usando <code>labels</code>, porque «flipper_length_mm» no es un texto para mostrarle a un cliente.</li>
    <li>Guarden la figura en <code>fig_escala</code> y muéstrenla.</li>
  </ul>
  Referencia: <a href="https://plotly.com/python/line-and-scatter/">plotly.com/python/line-and-scatter</a>.
</div>

In [ ]:
# Su código aquí


### Una tercera variable, y una sorpresa

`color` no es el único rol visual disponible. `size` mapea una variable numérica al tamaño del punto, y permite meter una tercera medida en el mismo scatter sin agregar otro eje.

<!-- WARNING -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210,153,34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ Este entregable va a fallar la primera vez</strong>
  El error es parte del ejercicio. Construyan el gráfico con <code>df</code> tal como está, ejecútenlo y <strong>lean el mensaje completo</strong> antes de decidir qué hacer. Con los datos válidos, el gráfico debería mostrar puntos cuyo tamaño represente la masa corporal.
</div>

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable 3 — Scatter con tamaño (0,2 puntos)</strong>
  Un scatter en <code>fig_size</code>, sobre <code>df</code>, con <code>flipper_length_mm</code> en X, <code>culmen_length_mm</code> en Y, color por especie y el <strong>tamaño del punto proporcional a <code>body_mass_g</code></strong>.
  <ul>
    <li>Ajusten el tamaño máximo del punto para que no se tapen entre sí; <code>px.scatter</code> tiene un argumento para eso.</li>
    <li>Etiqueten los ejes.</li>
    <li>Cuando resuelvan el error, <strong>dejen escrito en un comentario qué hicieron y sobre cuántas filas quedó el gráfico</strong>. No modifiquen <code>df</code>: el ajuste vale solo para esta figura.</li>
  </ul>
</div>

In [ ]:
# Su código aquí


### No exagerar con los ejes

En un gráfico de barras, recortar el eje Y puede exagerar las diferencias:
comparamos la longitud de las barras para interpretar sus valores.

In [ ]:
masa_por_especie = (
    df.group_by("species")
    .agg(pl.col("body_mass_g").mean())
    .sort("species")
)
masa_por_especie

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — el gráfico de barras</strong>
  Una <strong>barra</strong> compara una magnitud entre categorías, y lo hace por <em>longitud</em>: la barra que mide el doble representa el doble. Esa correspondencia es lo que la vuelve fácil de leer, y también lo que se rompe si el eje no parte del cero.
  <br><br>
  En <code>px</code> es <code>px.bar(tabla, x=..., y=...)</code>, normalmente sobre una tabla ya agregada. El rango de un eje se fija después, sobre la figura, con <code>update_yaxes(range=[...])</code>. Documentación: <a href="https://plotly.com/python/bar-charts/">plotly.com/python/bar-charts</a>.
</div>

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable 4 — La misma barra, dos ejes (0,2 puntos)</strong>
  Con <code>masa_por_especie</code>, construyan el mismo gráfico de barras dos veces:
  <ol>
    <li><code>fig_honesto</code>: tal cual sale de <code>px.bar</code>, sin tocar el eje.</li>
    <li><code>fig_exagerado</code>: idéntico, pero con el eje Y recortado para que arranque cerca de la barra más baja y termine cerca de la más alta.</li>
  </ol>
  Etiqueten el eje Y con su unidad y pónganle a cada figura un título que diga desde dónde parte su eje Y. Muestren las dos, una después de la otra, para poder compararlas de un vistazo.
</div>

In [ ]:
# Su código aquí


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 3 — Dos ejes, dos historias (0,2 puntos)</strong>
  Ambos gráficos usan los mismos tres promedios.
  <ol>
    <li>¿Cuál de los dos exagera la diferencia entre especies, y por qué?</li>
    <li>Si tuvieran que enviarle uno de los dos a alguien que va a tomar una decisión con eso, ¿cuál elegirían?</li>
  </ol>
</div>

`Escriban sus respuestas aquí:`

### Explorar antes de concluir

Hasta acá miraron gráficos sueltos. El **análisis exploratorio de datos** (EDA) es hacerlo de forma sistemática: cuántas observaciones y variables hay, de qué tipo, si faltan valores, cómo se distribuye cada una y cómo se relacionan entre sí.

Ya vieron dos señales de valores faltantes: el gráfico de los quince
colores omite dos pingüinos, y `size` no acepta esos registros. Revisemos
qué columnas tienen nulos y cuántos hay en cada una.

### Vista general

In [ ]:
print(f"Forma: {df.shape}")
df.select(pl.col(pl.Float64, pl.Int64)).describe()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 4 — Lo que dice el describe() (0,1 puntos)</strong>
  <ol>
    <li>El dataset tiene 344 filas, pero <code>count</code> marca 342 en varias columnas. ¿Qué les dice esa diferencia antes de haber revisado nada más?</li>
    <li>Revisen los <code>min</code> y <code>max</code> junto con las unidades de cada columna. ¿Hay valores negativos o señales evidentes de una unidad mal registrada? ¿Qué no podrían concluir sin información biológica de referencia?</li>
    <li>La tabla tiene siete columnas, pero <code>describe()</code> muestra cuatro. ¿Cuáles faltan y por qué? ¿Qué implica eso para lo que respondieron en el punto 2 de esta misma pregunta?</li>
  </ol>
</div>

`Escriban sus respuestas aquí:`

### Valores faltantes

`null_count()` cuenta los nulos de todas las columnas, incluidas las de
texto que excluimos del resumen anterior.

In [ ]:
print(df.null_count())

Los nulos son los valores que se marcaron con `NA` en el archivo. Una
columna de texto también puede traer valores inesperados que no estén
declarados como faltantes. Para detectarlos, hay que contar las categorías.

In [ ]:
print(df["sex"].value_counts().sort("count", descending=True))

<!-- WARNING -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210,153,34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ Un pingüino de sexo <code>.</code></strong>
  Hay una fila donde <code>sex</code> vale <code>"."</code>. No es un nulo: para <code>polars</code> es una categoría más, tan válida como <code>MALE</code>, porque al leer el archivo declaramos que el marcador de faltante era <code>NA</code>, y del punto nadie dijo nada.
  <br><br>
  <strong>No lo vamos a corregir en el dataset.</strong> Para decidir cómo tratar ese valor necesitaríamos saber qué significa y para qué se usará la tabla. Por ahora, cualquier gráfico o recuento que agrupe por <code>sex</code> mostrará una categoría de un solo pingüino.
</div>

### Análisis univariado

`px.histogram` sobre una columna de texto no corta ningún rango: cuenta
cuántas filas hay en cada categoría, así que sirve igual para ver el reparto
de `species`.

In [ ]:
fig_especies = px.histogram(df, x="species")
fig_especies.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 5 — ¿Balanceado? (0,1 puntos)</strong>
  <ol>
    <li>¿Hay cantidades similares de observaciones de las tres especies?</li>
    <li>Si calcularan el largo de aleta promedio de todo el dataset, ¿qué peso tendría cada especie en ese número? ¿Qué cambiaría si quisieran describir las tres especies por separado?</li>
  </ol>
</div>

`Escriban sus respuestas aquí:`

### Cómo se reparte el dataset

El histograma anterior cuenta una sola variable categórica. Para ver la
composición del dataset, agruparemos primero por especie, después por isla
y finalmente por sexo. Un gráfico jerárquico permite leer esos niveles en
ese orden.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — treemap y sunburst</strong>
  Los dos muestran lo mismo: cómo se reparte un total entre categorías anidadas, con el área de cada pieza proporcional a su tamaño. Cambian en la forma. El <strong>treemap</strong> usa rectángulos encajados, y aprovecha mejor el espacio cuando hay muchas categorías. El <strong>sunburst</strong> usa anillos concéntricos, y hace más evidente cuántos niveles tiene la jerarquía y a qué padre pertenece cada pieza.
  <br><br>
  En <code>px</code> el orden de agrupación se indica en <code>path</code>, desde el nivel más general hasta el más específico. <code>px.Constant("todos")</code> agrega una raíz común para los tres grupos de especies. Documentación: <a href="https://plotly.com/python/treemaps/">plotly.com/python/treemaps</a> y <a href="https://plotly.com/python/sunburst-charts/">plotly.com/python/sunburst-charts</a>.
</div>

<!-- WARNING -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210,153,34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ Este entregable también va a fallar la primera vez</strong>
  Construyan el gráfico con <code>df</code> tal como está, ejecútenlo y lean el error antes de modificar los datos. Con todos los registros representados, el gráfico debería mostrar la cantidad de pingüinos por especie, isla y sexo.
</div>

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable 5 — Treemap de la composición (0,2 puntos)</strong>
  Un treemap en <code>fig_treemap</code>, sobre <code>df</code>, con la jerarquía <strong>especie → isla → sexo</strong> y una raíz común. El área de cada pieza debe ser la cantidad de pingüinos.
  <ul>
    <li>Al resolver el error, <strong>no descarten filas</strong>: la solución pasa por darles a los nulos un nombre visible, no por sacarlos. <code>df</code> queda intacto.</li>
    <li>Después de dibujarlo, <strong>busquen el rectángulo más chico del gráfico</strong> y dejen escrito en un comentario qué encontraron ahí.</li>
  </ul>
</div>

In [ ]:
# Su código aquí


<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable 6 — El mismo reparto, en anillos (0,2 puntos)</strong>
  La misma jerarquía en <code>fig_sunburst</code>, ahora como sunburst. Reusen la tabla que prepararon para el treemap.
  <br><br>
  Después de verlos lado a lado, dejen en un comentario <strong>cuál de los dos elegirían para mandarle a Marisela y por qué</strong>. Una línea basta.
</div>

In [ ]:
# Su código aquí


### Cómo se distribuye cada medida

Los dos gráficos anteriores cuentan **cuántos** hay en cada grupo. Ahora
miraremos cómo se distribuye una medida numérica dentro de esos grupos.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — histograma y diagrama de caja</strong>
  Un <strong>histograma</strong> divide el rango de una variable numérica en intervalos y cuenta cuántas observaciones caen en cada uno. Permite ver dónde se concentran los valores y si aparecen uno o varios grupos; su aspecto depende de los intervalos elegidos. Documentación: <a href="https://plotly.com/python/histograms/">plotly.com/python/histograms</a>.
  <br><br>
  Un <strong>diagrama de caja</strong> muestra del primer al tercer cuartil, con la mediana marcada en el interior. El rango intercuartílico es la diferencia entre ambos cuartiles. Los bigotes llegan hasta los valores observados más extremos dentro de 1,5 veces ese rango desde los bordes de la caja; los valores que queden fuera se dibujan como puntos. Resume cada grupo sin mostrar la forma completa de su distribución. Documentación: <a href="https://plotly.com/python/box-plots/">plotly.com/python/box-plots</a>.
  <br><br>
  Usaremos el histograma para ver la forma de la distribución y las cajas para comparar medianas, cuartiles y posibles valores fuera de los bigotes entre especies.
</div>

Un **violín** muestra una estimación suavizada de la distribución: es más
ancho donde se concentran los valores. Como gráfico marginal, comparte el
eje de masa corporal con el histograma y permite comparar ambas vistas.
La suavización también puede ocultar detalles, así que conviene leerla
junto con las barras.

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable 7 — Histograma con margen (0,2 puntos)</strong>
  Un histograma de <code>body_mass_g</code> en <code>fig_masa_hist</code>, sobre <code>df</code>, coloreado por especie:
  <ul>
    <li>Prueben los modos de superposición que ofrece <code>barmode</code> y dejen el que permita ver las tres distribuciones a la vez; bajen la opacidad si se tapan.</li>
    <li>Agréguenle un <strong>gráfico marginal de violín</strong> sobre el eje X. <code>px.histogram</code> tiene un argumento para eso: está en <a href="https://plotly.com/python/marginal-plots/">plotly.com/python/marginal-plots</a>.</li>
    <li>Etiqueten los ejes.</li>
  </ul>
</div>

In [ ]:
# Su código aquí


<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable 8 — La misma masa, en cajas (0,2 puntos)</strong>
  Un diagrama de caja en <code>fig_masa_caja</code>, con una caja por especie y la masa corporal en el eje Y. Etiqueten los ejes.
  <br><br>
  Comparen las tres vistas que llevan de la misma variable —el histograma, su violín marginal y estas cajas— y dejen en un comentario <strong>qué muestra la caja que el histograma no</strong>.
</div>

In [ ]:
# Su código aquí


### Análisis bivariado

<!-- WARNING -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210,153,34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ <code>DataFrame.corr()</code> no sirve acá</strong>
  Con nulos adentro, <code>df.select(numericas).corr()</code> devuelve una matriz <strong>entera de <code>NaN</code></strong>: no descarta los faltantes, los propaga. La expresión <code>pl.corr(a, b)</code> sí los descarta, y lo hace <em>por par</em>, usando las 342 filas donde ambas columnas están medidas.
  <br><br>
  Es la diferencia entre una función que asume datos limpios y una que no. Por eso la matriz se arma par por par.
</div>

La **correlación de Pearson** resume la dirección y la intensidad de una
asociación lineal entre dos medidas. Toma valores de −1 a 1: el signo indica
la dirección y el valor absoluto indica qué tan fuerte es la asociación
lineal. Un valor cercano a cero no descarta otras formas de relación. Al
interpretar la matriz, recuerden que mezcla las tres especies.

In [ ]:
numericas = [
    "culmen_length_mm",
    "culmen_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]

# Una fila por par de variables, que después se pivotea a matriz cuadrada.
pares = pl.DataFrame(
    {
        "variable": [a for a in numericas for _ in numericas],
        "contra": [b for _ in numericas for b in numericas],
        "correlacion": [
            df.select(pl.corr(a, b)).item()
            for a in numericas
            for b in numericas
        ],
    }
).with_columns(pl.col("correlacion").round(2))

corr = pares.pivot(on="contra", index="variable", values="correlacion")
corr

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — <code>px.imshow</code> y los mapas de calor</strong>
  <code>px.imshow</code> dibuja una matriz como una grilla de celdas coloreadas por su valor. Es la forma habitual de leer una matriz de correlación: los pares fuertes saltan por color, sin tener que recorrer números uno por uno.
  <br><br>
  A diferencia de los gráficos anteriores, no recibe columnas sino la matriz completa, y espera que las etiquetas de filas y columnas vayan aparte. Documentación: <a href="https://plotly.com/python/imshow/">plotly.com/python/imshow</a>, y <a href="https://plotly.com/python/heatmaps/">plotly.com/python/heatmaps</a> para la familia completa.
  <br><br>
  Una correlación va de −1 a 1 y el cero significa algo: no es un extremo, es el centro. Eso pide una escala <strong>divergente</strong>, con un color a cada lado y un tono neutro al medio, y un rango fijado a mano en vez del que la librería deduzca de los datos.
</div>

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable 9 — La matriz como mapa de calor (0,2 puntos)</strong>
  Dibujen la matriz <code>corr</code> con <code>px.imshow</code>, en <code>fig_corr</code>:
  <ul>
    <li>La columna <code>variable</code> trae las etiquetas, no un dato: sáquenla antes de pasar la matriz y úsenla como <code>x</code> e <code>y</code>.</li>
    <li>Escala divergente, y el rango fijado en −1 a 1. Busquen en la referencia de la API qué argumentos hacen esas dos cosas.</li>
    <li>Muestren el valor numérico dentro de cada celda: <code>px.imshow</code> tiene un argumento para eso y se llega a él desde la galería.</li>
  </ul>
  Referencia de la API: <a href="https://plotly.com/python-api-reference/generated/plotly.express.imshow.html">plotly.express.imshow</a>.
</div>

In [ ]:
# Su código aquí


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — facetas</strong>
  Una <strong>faceta</strong> parte un gráfico en varios paneles pequeños, uno por categoría, todos con los mismos ejes. En <code>px</code> se pide con <code>facet_col</code> o <code>facet_row</code>.
  <br><br>
  Es la alternativa a meter una variable más como color en el mismo panel, y sirve cuando los grupos se superponen tanto que el color ya no alcanza para separarlos. Como los ejes se comparten, los paneles se pueden comparar entre sí directamente. Documentación: <a href="https://plotly.com/python/facet-plots/">plotly.com/python/facet-plots</a>.
  <br><br>
  Es el cuarto principio de la Parte 1 en acción: antes de amontonar variables en un gráfico, evaluar si conviene hacer varios.
</div>

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable 10 — La misma relación, especie por especie (0,2 puntos)</strong>
  En la matriz conjunta, <code>culmen_length_mm</code> y <code>culmen_depth_mm</code> tienen una correlación lineal de −0,24. Antes de interpretarla, miren los datos por especie.
  <ul>
    <li>Un <code>px.scatter</code> con esas dos variables, en <code>fig_culmen</code>, con <strong>un panel por especie</strong> y cada especie de un color.</li>
    <li>Agreguen la línea de tendencia de cada panel con <code>trendline="ols"</code>. Con los mismos ejes, podrán comparar la dirección de las pendientes entre paneles. La pendiente, por sí sola, no mide la intensidad de la correlación.</li>
    <li>Etiqueten los ejes.</li>
  </ul>
  Miren los tres paneles antes de responder la pregunta siguiente.
</div>

In [ ]:
# Su código aquí


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 6 — Correlación conjunta y por especie (0,2 puntos)</strong>
  <ol>
    <li>Según el mapa de calor, ¿qué par de variables numéricas está más correlacionado? ¿Tiene sentido biológico esa relación?</li>
    <li>La matriz conjunta da −0,24 entre <code>culmen_length_mm</code> y <code>culmen_depth_mm</code>. Miren las tres facetas y las líneas de tendencia. ¿En qué dirección cambia el alto del pico al aumentar el largo <strong>dentro</strong> de cada especie?</li>
    <li>¿Qué les dice la diferencia entre esos dos resultados sobre calcular una correlación con las tres especies mezcladas?</li>
  </ol>
</div>

`Escriban sus respuestas aquí:`

---
## Parte 2 — Reportería con Streamlit (3,0 puntos)

Vamos a llevar parte de la exploración a un panel de Streamlit. Desde el
navegador se podrá filtrar la tabla y consultar los gráficos sin ejecutar
celdas del notebook.

En esta parte construirán el panel completo. Los requisitos y la
comprobación aparecen más abajo.

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 Un panel para revisar los resultados</strong>
  Marisela revisó la exploración de los pingüinos. Antes de mostrarla en una reunión, quiere un panel local en el que otra persona pueda ordenar y filtrar la tabla, ver los gráficos y entender qué registros tienen problemas de calidad.
  <br><br>
  «Si un número llama la atención, quiero poder encontrar los registros detrás de él y saber qué datos faltan», les dice. Además, les pide que definan colores propios para distinguir el panel de la configuración inicial de Streamlit; pueden elegirlos ustedes.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — reportería</strong>
  La <strong>reportería</strong> consiste en seleccionar, organizar y presentar resultados de un análisis para que otras personas puedan consultarlos. En este laboratorio el resultado será un panel: sus gráficos deben responder preguntas concretas, los filtros deben permitir revisar distintos grupos de pingüinos y el informe de calidad debe explicar qué límites tienen las cifras.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Lo fundamental de Streamlit</strong>
  Un panel de Streamlit es un archivo <code>.py</code> normal que se ejecuta de arriba hacia abajo — pero cada vez que alguien mueve un control, <strong>todo el archivo se vuelve a ejecutar</strong>: cada una de esas reejecuciones es un <em>rerun</em>. Con esa idea en mente, las piezas básicas son:
  <ul>
    <li><code>st.title</code>, <code>st.subheader</code>, <code>st.caption</code>: texto y estructura.</li>
    <li><code>st.sidebar</code>, <code>st.columns</code>: dónde va cada cosa en la pantalla.</li>
    <li><code>st.selectbox</code>, <code>st.multiselect</code>, <code>st.slider</code>: los controles con los que alguien filtra.</li>
    <li><code>st.dataframe</code>, <code>st.metric</code>, <code>st.plotly_chart</code>: cómo se muestra una tabla, un número o un gráfico.</li>
    <li><code>@st.cache_data</code>: evita releer o recalcular algo que no cambió entre un rerun y el siguiente — típicamente, cargar el dataset.</li>
  </ul>
  Se ejecuta con <code>streamlit run archivo.py</code>, nunca importándolo ni corriéndolo como celda de notebook.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — el tema de la aplicación</strong>
  Streamlit lee su configuración de <code>.streamlit/config.toml</code>, dentro de la carpeta desde la que se ejecuta el panel. La tabla <code>[theme]</code> controla cómo se ve: <code>base</code> elige el tema de partida —<code>"light"</code> o <code>"dark"</code>—, y <code>primaryColor</code>, <code>backgroundColor</code>, <code>secondaryBackgroundColor</code> y <code>textColor</code> permiten reemplazar sus colores.
  <br><br>
  <code>[theme.light]</code> y <code>[theme.dark]</code> permiten definir variantes para cada modo. <code>[theme.sidebar]</code> configura la barra lateral por separado.
  <br><br>
  <code>st.plotly_chart</code> usa <code>theme="streamlit"</code> por omisión, de modo que los gráficos adoptan los colores del panel. Con <code>theme=None</code> se conserva el estilo de Plotly.
  <br><br>
  Documentación: <a href="https://docs.streamlit.io/develop/concepts/configuration/theming">guía de temas</a>, <a href="https://docs.streamlit.io/develop/concepts/configuration/theming-customize-colors-and-borders">colores y bordes</a> y la <a href="https://docs.streamlit.io/develop/api-reference/configuration/config.toml">referencia de config.toml</a>.
</div>

Un `.streamlit/config.toml` mínimo se ve así:

```toml
[theme]
base = "dark"
primaryColor = "#0f9e99"
textColor = "#fafafa"

[theme.sidebar]
primaryColor = "#f0883e"
```

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — la tabla interactiva</strong>
  <code>st.dataframe</code> no es una imagen de una tabla: quien la abre puede <strong>ordenar</strong> por cualquier columna haciendo clic en su encabezado, y <strong>buscar</strong> texto dentro de ella. Eso viene de fábrica y no hay que programarlo.
  <br><br>
  Lo que <strong>no</strong> trae es filtrar por columna. El filtrado se arma con widgets —<code>st.multiselect</code> para categorías, <code>st.slider</code> para rangos— que devuelven la selección, y con ella se filtra la tabla antes de pasársela a <code>st.dataframe</code>. Como el panel se reejecuta entero con cada cambio, la tabla se redibuja sola.
  <br><br>
  <code>st.column_config</code> controla cómo se muestra cada columna: el nombre visible, el formato de un número, su ancho, un tooltip de ayuda.
  <br><br>
  Documentación: <a href="https://docs.streamlit.io/develop/api-reference/data/st.dataframe">st.dataframe</a> y <a href="https://docs.streamlit.io/develop/api-reference/data/st.column_config">st.column_config</a>.
</div>

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Entregable — su panel de pingüinos (3,0 puntos)</strong>
  Construyan un panel de Streamlit sobre <strong>el mismo dataset de la Parte 1</strong>. En la raíz del laboratorio tienen <code>mi_panel.py</code>, con los imports, la configuración de la página y el cargador ya escritos —eso es infraestructura y no se evalúa— y las cuatro secciones vacías. Pueden renombrar el archivo y ordenar las secciones como prefieran. El panel debe incluir:
  <ol>
    <li>una <strong>tabla interactiva</strong> que se pueda ordenar por columna y filtrar con al menos dos controles: uno categórico y uno de rango numérico. Los mismos filtros deben afectar también a los gráficos. Denle formato a las columnas con <code>st.column_config</code>. Un registro sin valor en la columna del filtro numérico queda fuera de la selección filtrada; si no queda ninguna fila, muestren un aviso en la página;</li>
    <li>un informe de <strong>calidad de los datos</strong> visible en la página y calculado siempre sobre el CSV completo, aunque se apliquen filtros: indiquen qué columnas tienen nulos y cuántos, identifiquen el valor inesperado de <code>sex</code> y expliquen cómo pueden afectar los recuentos, filtros o gráficos. Calculen las cifras desde el dataset, sin escribirlas a mano;</li>
    <li>al menos <strong>cuatro gráficos</strong> a elección, cada uno con una explicación en la página de por qué esa información es útil y por qué eligieron esa visualización. Pueden reusar los de la Parte 1 o construir otros; lo que no vale es entregar cuatro veces el mismo tipo;</li>
    <li>un <strong>tema propio</strong> en <code>.streamlit/config.toml</code>. No sirve dejar el de fábrica: tienen que decidir colores y dejar el archivo en la entrega.</li>
  </ol>
  Conserven el dataset original. Los ajustes necesarios para un gráfico se hacen en una tabla aparte y se explican; el informe describe los problemas del CSV sin corregirlos.
</div>

<table>
  <tr><th>Criterio</th><th>Puntaje</th></tr>
  <tr><td>Tabla interactiva con formato; dos filtros aplicados a tabla y gráficos; aviso si la selección queda vacía</td><td>0,6</td></tr>
  <tr><td>Nulos y valor inesperado de <code>sex</code>, con cifras calculadas y consecuencias explicadas</td><td>0,4</td></tr>
  <tr><td>Cuatro gráficos, cada uno con su justificación (0,4 c/u)</td><td>1,6</td></tr>
  <tr><td>Tema propio en config.toml</td><td>0,4</td></tr>
</table>

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Antes de entregar</strong>
  Desde la raíz del laboratorio, donde están el archivo del panel y la carpeta <code>.streamlit</code>, abran el panel con <code>uv run streamlit run &lt;archivo&gt;.py</code> y revisen que se cumpla lo siguiente:
  <ul>
    <li>borraron el <code>raise NotImplementedError</code> del andamiaje;</li>
    <li>la terminal no muestra ningún error ni ninguna advertencia de deprecación, y la página no muestra ningún recuadro rojo;</li>
    <li>el panel <strong>no se ve como el de fábrica</strong>: si renombran su <code>config.toml</code> y reinician el panel, tiene que verse distinto —y después devuélvanle el nombre—;</li>
    <li>al hacer clic en el encabezado de una columna, la tabla se reordena; al mover los dos filtros, cambian la tabla y los gráficos; si la selección queda vacía, aparece un aviso;</li>
    <li>se cuentan cuatro gráficos de al menos dos tipos distintos, cada uno con su explicación <strong>visible en la página</strong> — no en un comentario del código, que nadie abre;</li>
    <li>el informe de calidad muestra los nulos y el valor inesperado de <code>sex</code>, con cifras calculadas desde el CSV completo y una explicación de sus consecuencias; los filtros no cambian esas cifras.</li>
  </ul>
</div>

---
## Entrega

Cualquier duda, al canal del curso en Discord o al correo del equipo docente.
Confirmen la fecha y hora de entrega en el anuncio del laboratorio.